In [2]:
import cv2
import os
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from skimage.feature import hog
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix


In [3]:
def load_images_folder_histogram(directory):
    data = []
    labels = []
    for subdir in os.listdir(directory):
        subdir_path = os.path.join(directory, subdir)
        if os.path.isdir(subdir_path):
            for filename in os.listdir(subdir_path):
                filepath = os.path.join(subdir_path, filename)
                if os.path.isfile(filepath):
                    image = cv2.imread(filepath, cv2.IMREAD_GRAYSCALE)  
                    image = cv2.resize(image, (64, 64))  
                    arr = (image.flatten()) 
                    arr1 = np.histogram(arr, 10)
                    data.append(arr1[0])
                    if subdir == "dogs":
                        labels.append(0)
                    elif subdir == "cats":
                        labels.append(1)
                    else:
                        labels.append(2)
    return np.array(data), np.array(labels)  

In [4]:
def load_images_folder_mmm(directory):
    data = []
    labels = []
    for subdir in os.listdir(directory):
        subdir_path = os.path.join(directory, subdir)
        if os.path.isdir(subdir_path):
            for filename in os.listdir(subdir_path):
                filepath = os.path.join(subdir_path, filename)
                if os.path.isfile(filepath):
                    image = cv2.imread(filepath, cv2.IMREAD_GRAYSCALE)
                    image = cv2.resize(image, (64, 64))  
                    arr = image.flatten() 
                    arr1 = [np.average(arr), np.max(arr), np.median(arr)]  
                    data.append(arr1)
                    if subdir == 'dogs':
                        labels.append(0)
                    elif subdir == 'cats':
                        labels.append(1)
                    else :labels.append(2)
    return np.array(data), np.array(labels)  

In [5]:
def load_images_folder_gabor(directory):
    data = []
    labels = []
    ksize = 5  
    sigma = 1.0
    lambd = 10.0
    gamma = 0.5
    thetas = [0, 45, 90, 135]  

    for subdir in os.listdir(directory):
        subdir_path = os.path.join(directory, subdir)
        if os.path.isdir(subdir_path):
            for filename in os.listdir(subdir_path):
                filepath = os.path.join(subdir_path, filename)
                if os.path.isfile(filepath):
                    image = cv2.imread(filepath, cv2.IMREAD_GRAYSCALE)
                    image = cv2.resize(image, (64, 64))  
                    gabor_features = []
                    for theta in thetas:
                        theta_rad = np.deg2rad(theta)
                        kernel = cv2.getGaborKernel((ksize, ksize), sigma, theta_rad, lambd, gamma, 0, ktype=cv2.CV_32F)
                        filtered_img = cv2.filter2D(image, cv2.CV_8UC3, kernel)  
                        gabor_features.append(np.mean(filtered_img))  
                        gabor_features.append(np.var(filtered_img))   
                    data.append(gabor_features)  
                    if subdir == 'dogs':
                        labels.append(0)
                    elif subdir == 'cats':
                        labels.append(1)
                    else :labels.append(2)
    return np.array(data), np.array(labels)  

In [6]:
def load_images_folder_hog(directory):
    data = []
    labels = []
    for subdir in os.listdir(directory):
        subdir_path = os.path.join(directory, subdir)
        if os.path.isdir(subdir_path):
            for filename in os.listdir(subdir_path):
                filepath = os.path.join(subdir_path, filename)
                if os.path.isfile(filepath):
                    image = cv2.imread(filepath, cv2.IMREAD_GRAYSCALE)
                    image = cv2.resize(image, (64, 64)) 
                    hog_features = hog(image, orientations=9, pixels_per_cell=(8, 8),
                                       cells_per_block=(2, 2), block_norm='L2-Hys', feature_vector=True)
                    data.append(hog_features)
                    if subdir == 'dogs':
                        labels.append(0)
                    elif subdir == 'cats':
                        labels.append(1)
                    else :labels.append(2)
    return np.array(data), np.array(labels)  

In [ ]:

dataset_directory1 = r"C:/D folder/Programming/programs/my codes/ML codes/knn classifier/Dog-Cat Dataset/train"
dataset_directory2 = r"C:/D folder/Programming/programs/my codes/ML codes/knn classifier/Dog-Cat Dataset/test"

images_train1, labels_train1 = load_images_folder_hog(dataset_directory1)
images_test1, labels_test1 = load_images_folder_hog(dataset_directory2)

images_train2, labels_train2 = load_images_folder_histogram(dataset_directory1)
images_test2, labels_test2 = load_images_folder_histogram(dataset_directory2)

images_train3, labels_train3 = load_images_folder_gabor(dataset_directory1)
images_test3, labels_test3 = load_images_folder_gabor(dataset_directory2)
scaler = StandardScaler()
images_train1 = scaler.fit_transform(images_train1)
images_test1 = scaler.transform(images_test1)

images_train2 = scaler.fit_transform(images_train2)
images_test2 = scaler.transform(images_test2)

images_train3 = scaler.fit_transform(images_train3)
images_test3 = scaler.transform(images_test3)


In [ ]:

print(images_test1.shape)
print(images_test2.shape)
print(images_test3.shape)

(230, 3)


In [14]:
acc1 = []
# for i in np.arange(1, 100,2):
    # for j in np.arange(0.01, 1.1, 0.1):  
# svm_linear  = SVC(kernel='linear', C=0.5)  
# svm_rbf     = SVC(kernel='rbf', C=0.5, gamma='scale')  
svm_poly1    = SVC(kernel='poly', C=1.0, degree=3, gamma='scale')   
# svm_sigmoid = SVC(kernel='sigmoid', C=0.5, gamma='scale', coef0=0) 
svm_poly1.fit(images_train1, labels_train1)
y_pred1 = svm_poly1.predict(images_test1)
accuracy = accuracy_score(labels_test1, y_pred1)
acc1.append(accuracy * 100)  
best_acc1 = max(acc1)
print(f"Best Accuracy: {best_acc1:.2f}%")

print(confusion_matrix(labels_test1, y_pred1))
print(classification_report(labels_test1,y_pred1))

Best Accuracy: 43.91%
[[13  0 57]
 [13  4 53]
 [ 4  2 84]]
              precision    recall  f1-score   support

           0       0.43      0.19      0.26        70
           1       0.67      0.06      0.11        70
           2       0.43      0.93      0.59        90

    accuracy                           0.44       230
   macro avg       0.51      0.39      0.32       230
weighted avg       0.50      0.44      0.34       230



In [10]:
acc2 = []
# for i in np.arange(1, 100,2):
    # for j in np.arange(0.01, 1.1, 0.1):  
# svm_linear  = SVC(kernel='linear', C=0.5)  
# svm_rbf     = SVC(kernel='rbf', C=0.5, gamma='scale')  
svm_poly2    = SVC(kernel='poly', C=1.0, degree=3, gamma='scale')   
# svm_sigmoid = SVC(kernel='sigmoid', C=0.5, gamma='scale', coef0=0) 
svm_poly2.fit(images_train2, labels_train2)
y_pred2 = svm_poly2.predict(images_test2)
accuracy = accuracy_score(labels_test2, y_pred2)
acc2.append(accuracy * 100)  
best_acc2 = max(acc2)
print(f"Best Accuracy: {best_acc2:.2f}%")

print(confusion_matrix(labels_test2, y_pred2))
print(classification_report(labels_test2,y_pred2))

Best Accuracy: 41.74%
[[ 3 18 49]
 [10 16 44]
 [ 1 12 77]]
              precision    recall  f1-score   support

           0       0.21      0.04      0.07        70
           1       0.35      0.23      0.28        70
           2       0.45      0.86      0.59        90

    accuracy                           0.42       230
   macro avg       0.34      0.38      0.31       230
weighted avg       0.35      0.42      0.34       230



In [11]:
acc3 = []
# for i in np.arange(1, 100,2):
    # for j in np.arange(0.01, 1.1, 0.1):  
# svm_linear  = SVC(kernel='linear', C=0.5)  
# svm_rbf     = SVC(kernel='rbf', C=0.5, gamma='scale')  
svm_poly3    = SVC(kernel='poly', C=1.0, degree=3, gamma='scale')   
# svm_sigmoid = SVC(kernel='sigmoid', C=0.5, gamma='scale', coef0=0) 
svm_poly3.fit(images_train3, labels_train3)
y_pred3 = svm_poly3.predict(images_test3)
accuracy = accuracy_score(labels_test3, y_pred3)
acc3.append(accuracy * 100)  
best_acc3 = max(acc3)
print(f"Best Accuracy: {best_acc3:.2f}%")

print(confusion_matrix(labels_test3, y_pred3))
print(classification_report(labels_test3,y_pred3))

Best Accuracy: 38.70%
[[ 0  0 70]
 [ 2  0 68]
 [ 0  1 89]]
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        70
           1       0.00      0.00      0.00        70
           2       0.39      0.99      0.56        90

    accuracy                           0.39       230
   macro avg       0.13      0.33      0.19       230
weighted avg       0.15      0.39      0.22       230

